# Retrieval Pipelines

**Thesis:** RAG-Driven Natural Language Energy Demand Forecasting  
**Author:** Zoheb Anwar Hussain (Student ID: 1196931)

---

### What this notebook does
Runs all 50 golden dataset queries through each query's targeted retrieval
pipeline and records the retrieved documents for evaluation in Phase 6.

### Three Pipelines
| Pipeline | Strategy | LangChain Components |
|----------|----------|---------------------|
| 1. Dense | FAISS cosine similarity | `FAISS.as_retriever()` |
| 2. Hybrid | BM25 + FAISS fusion | `BM25Retriever` + `EnsembleRetriever` |
| 3. Hierarchical | FAISS + parent_id expansion | FAISS + Python parent lookup |

### Evaluation Strategy
**Option B** — Each query runs through its targeted pipeline + a baseline
dense run. The full Option A comparison (all queries × all pipelines) can
be run later by changing one loop.

### No API Calls
Retrieval runs entirely locally. No Gemini, no Groq.

---
## Cell 1 — Imports and Setup

In [5]:
%load_ext autoreload
%autoreload 2

import json
import pandas as pd
from pathlib import Path

from config import PATHS
from config.paths import create_all_directories
from src.utils import setup_logger, log_section
from src.embedding import (
    load_kb_documents,
    get_embeddings_model,
    load_faiss_index,
)
from src.retrieval import (
    DenseRetriever,
    HybridRetriever,
    HierarchicalRetriever,
)

logger = setup_logger("retrieval")
create_all_directories()
logger.info("All imports successful.")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


2026-04-29 20:59:10 | INFO     | retrieval | All imports successful.


---
## Cell 2 — Load FAISS Index and KB Documents

In [6]:
log_section("Loading FAISS index and KB documents", 1, 5)

# Load embedding model (same one used to build the index)
embeddings = get_embeddings_model()

# Load FAISS index from disk
faiss_vs = load_faiss_index(PATHS["faiss_index"], embeddings)

# Load documents (needed by BM25 and hierarchical parent lookup)
kb_csv_path = PATHS["summaries_csv"] / "combined_master_summaries.csv"
documents = load_kb_documents(kb_csv_path)

logger.info(
    "Loaded FAISS index and %d KB documents.", len(documents)
)


  STEP 1/5  [█░░░░] 20%
  Loading FAISS index and KB documents



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

2026-04-29 20:59:24 | INFO     | retrieval | Loaded FAISS index and 140 KB documents.


---
## Cell 3 — Initialise All Three Retrievers

In [7]:
log_section("Initialising retrievers", 2, 5)

K = 5  # Number of documents to retrieve per query

# Pipeline 1 — Dense (baseline)
dense_retriever = DenseRetriever(faiss_vs, k=K)

# Pipeline 2 — Hybrid (BM25 + FAISS)
hybrid_retriever = HybridRetriever(
    faiss_vs, documents, k=K, weights=[0.3, 0.7]
)

# Pipeline 3 — Hierarchical (FAISS + parent expansion)
hierarchical_retriever = HierarchicalRetriever(
    faiss_vs, documents, k=K
)

# Map pipeline names to retriever instances
RETRIEVERS = {
    "dense":        dense_retriever,
    "hybrid":       hybrid_retriever,
    "hierarchical": hierarchical_retriever,
}

logger.info("All 3 retrievers initialised.")

2026-04-29 20:59:36 | INFO     | retrieval | All 3 retrievers initialised.



  STEP 2/5  [██░░░] 40%
  Initialising retrievers



---
## Cell 4 — Load Golden Dataset and Run Retrieval

**Option B strategy:** Each query runs through:
1. Its targeted pipeline (from `retrieval_strategy_target`)
2. Dense baseline (always)

Queries with `retrieval_strategy_target = 'all'` run through all 3 pipelines.

In [8]:
log_section("Running retrieval for all golden queries", 3, 5)

# Load golden dataset
golden_path = PATHS["golden_dataset"] / "combined_golden_dataset.csv"
golden_df = pd.read_csv(golden_path)
logger.info("Golden dataset loaded: %d queries.", len(golden_df))

# Results accumulator
retrieval_results = []

for _, row in golden_df.iterrows():
    query      = row["user_query"]
    golden_id  = row["golden_id"]
    target     = row["retrieval_strategy_target"]
    expected   = json.loads(row["expected_summary_ids"])

    # Determine which pipelines to run for this query
    if target == "all":
        pipelines_to_run = ["dense", "hybrid", "hierarchical"]
    else:
        # Targeted pipeline + dense baseline (always)
        pipelines_to_run = list(set(["dense", target]))

    for pipeline_name in pipelines_to_run:
        retriever = RETRIEVERS[pipeline_name]
        docs = retriever.retrieve(query)

        retrieved_ids = [
            doc.metadata.get("source", "") for doc in docs
        ]

        retrieval_results.append({
            "golden_id":           golden_id,
            "pipeline":            pipeline_name,
            "query":               query,
            "target_pipeline":     target,
            "expected_ids":        json.dumps(expected),
            "retrieved_ids":       json.dumps(retrieved_ids),
            "num_retrieved":       len(retrieved_ids),
            "num_expected":        len(expected),
        })

results_df = pd.DataFrame(retrieval_results)

# Save to disk
output_path = PATHS["retrieval_results"] / "retrieval_results.csv"
results_df.to_csv(output_path, index=False)

logger.info(
    "Retrieval complete: %d query-pipeline combinations saved to %s.",
    len(results_df), output_path,
)

# Summary
print(f"\nTotal retrieval runs: {len(results_df)}")
print(results_df["pipeline"].value_counts().to_string())

2026-04-29 20:59:48 | INFO     | retrieval | Golden dataset loaded: 50 queries.



  STEP 3/5  [███░░] 60%
  Running retrieval for all golden queries



2026-04-29 20:59:51 | INFO     | retrieval | Retrieval complete: 93 query-pipeline combinations saved to e:\UpGrad\LJMU_Thesis\Topic_1_Energy_Forecasting\Research_Proposal\Git_Repo\RAG-Based-Energy-Forecasting\notebooks\outputs\retrieval_results\retrieval_results.csv.



Total retrieval runs: 93
pipeline
dense           50
hybrid          23
hierarchical    20


---
## Cell 5 — Quick Retrieval Quality Check

Compute hit rate (did at least one expected document appear in retrieved results?)
per pipeline as a quick sanity check before moving to full RAGAS evaluation.

In [9]:
log_section("Quick retrieval quality check", 4, 5)

def compute_hit_rate(row):
    """Check if at least one expected ID was retrieved."""
    expected  = set(json.loads(row["expected_ids"]))
    retrieved = set(json.loads(row["retrieved_ids"]))
    return 1 if expected & retrieved else 0

results_df["hit"] = results_df.apply(compute_hit_rate, axis=1)

print("\n" + "=" * 55)
print("  HIT RATE BY PIPELINE (at least 1 expected doc retrieved)")
print("=" * 55)

for pipeline in ["dense", "hybrid", "hierarchical"]:
    subset = results_df[results_df["pipeline"] == pipeline]
    if subset.empty:
        continue
    hit_rate = subset["hit"].mean() * 100
    print(
        f"  {pipeline:<15} {hit_rate:>5.1f}% "
        f"({subset['hit'].sum()}/{len(subset)} queries)"
    )

print("\n" + "=" * 55)


  STEP 4/5  [████░] 80%
  Quick retrieval quality check


  HIT RATE BY PIPELINE (at least 1 expected doc retrieved)
  dense            26.0% (13/50 queries)
  hybrid            8.7% (2/23 queries)
  hierarchical     40.0% (8/20 queries)



---
## Cell 6 — Sample Retrieval Results

Preview retrieved documents for a few queries to visually verify relevance.

In [10]:
log_section("Sample retrieval results", 5, 5)

# Show 3 sample queries — one from each pipeline
for pipeline_name in ["dense", "hybrid", "hierarchical"]:
    subset = results_df[results_df["pipeline"] == pipeline_name]
    if subset.empty:
        continue

    sample = subset.iloc[0]
    print(f"\n{'=' * 60}")
    print(f"  Pipeline: {pipeline_name.upper()}")
    print(f"  Query: {sample['query'][:80]}...")
    print(f"  Expected IDs: {sample['expected_ids'][:80]}...")
    print(f"  Retrieved IDs:")

    retrieved = json.loads(sample["retrieved_ids"])
    expected  = set(json.loads(sample["expected_ids"]))
    for rid in retrieved:
        match = "MATCH" if rid in expected else "     "
        print(f"    [{match}] {rid}")

print(f"\n{'=' * 60}")


  STEP 5/5  [█████] 100%
  Sample retrieval results


  Pipeline: DENSE
  Query: What was the average daily electricity load and peak load recorded across GEFCom...
  Expected IDs: ["gefcom_daily_18_2005-08-07", "gefcom_daily_3_2005-08-07", "gefcom_daily_10_200...
  Retrieved IDs:
    [     ] gefcom_system_level_weekly_W3_2004
    [     ] gefcom_monthly_11.0_August_2005
    [     ] gefcom_weekly_19.0_W17.0_2005.0
    [     ] gefcom_monthly_19.0_August_2005
    [MATCH] gefcom_daily_19_2005-08-07

  Pipeline: HYBRID
  Query: What is the total daily energy consumption in MWh typically observed across GEFC...
  Expected IDs: ["gefcom_daily_18_2005-08-07", "gefcom_daily_3_2005-08-07", "gefcom_daily_10_200...
  Retrieved IDs:
    [     ] gefcom_daily_11_2005-08-07
    [MATCH] gefcom_daily_19_2005-08-07
    [MATCH] gefcom_daily_1_2005-08-07
    [MATCH] gefcom_daily_10_2007-11-29
    [     ] gefcom_daily_8_2007-11-29

  Pipeline: HIERARCHICAL
  Query: What weekly demand patterns are observed 

---
## Pipeline Complete

Retrieval results saved to:
```
outputs/retrieval_results/retrieval_results.csv
```

Columns: `golden_id`, `pipeline`, `query`, `target_pipeline`,
`expected_ids`, `retrieved_ids`, `num_retrieved`, `num_expected`

### Option A Upgrade (for later)
To run ALL 50 queries through ALL 3 pipelines, change Cell 4:
```python
# Replace this:
if target == 'all':
    pipelines_to_run = ['dense', 'hybrid', 'hierarchical']
else:
    pipelines_to_run = list(set(['dense', target]))

# With this:
pipelines_to_run = ['dense', 'hybrid', 'hierarchical']  # Always all 3
```

### Next Phase
Move to `notebooks/05_rag_generation.ipynb` to generate RAG answers
using Llama 3.3 70B via Groq for each retrieved context.